# Can we predict hospital readmission?
Dataset: `data.csv` (diabetic inpatient encounters; target `readmitted` in {`<30`, `>30`, `NO`}).

Key design decisions:
- **`patient_nbr` has several rows per patient**, so every split (hold-out and CV) is grouped by patient. Otherwise the same patient lands in train and test and the score is inflated.
- Two binary framings: **A)** readmitted within 30 days (`<30`) vs. not, the usual operational target; **B)** readmitted at all (`<30` or `>30`) vs. `NO`.
- Encounters ending in death or hospice cannot be readmitted, so they are removed.
- Metrics are ROC-AUC and PR-AUC (the `<30` class is ~11%, so accuracy is misleading), compared against a prevalence baseline.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv',low_memory=False)
print(df.shape, 'patients:',df.patient_nbr.nunique())
df.readmitted.value_counts(normalize=True)

## Data checks

In [ ]:
print('duplicate encounters:',df.encounter_id.duplicated().sum())
print('encounters per patient:'); print(df.groupby('patient_nbr').size().describe()[['mean','50%','max']])
miss=df.isna().mean(); print(miss[miss>0].sort_values(ascending=False).round(3))
print(df.discharge_disposition_id.value_counts().head(10))

## Preparation
- Drop identifiers (`patient_nbr` is kept only as the split group), `weight` (97% missing), `examide`/`citoglipton` (constant).
- Remove death/hospice discharges (ids 11, 13, 14, 19, 20, 21).
- Treat admission/discharge codes as categories. Collapse ICD-9 diagnosis codes to broad groups. `age` bracket becomes its numeric midpoint.
- Missing values become an explicit `Missing` category (missingness of payer/specialty may itself be informative).

In [ ]:
d=df[~df.discharge_disposition_id.isin([11,13,14,19,20,21])].copy()
print('removed',len(df)-len(d),'death/hospice encounters')
d=d[d.gender!='Unknown/Invalid'].reset_index(drop=True)
groups=d.patient_nbr.values
d['y30']=(d.readmitted=='<30').astype(int)
d['yany']=(d.readmitted!='NO').astype(int)

def icd_group(x):
    if pd.isna(x): return 'Missing'
    x=str(x)
    if x[0] in 'VE': return 'V/E'
    v=float(x)
    if 250<=v<251: return 'Diabetes'
    for lo,hi,n in [(390,460,'Circulatory'),(460,520,'Respiratory'),(520,580,'Digestive'),(580,630,'Genitourinary'),
                    (140,240,'Neoplasm'),(710,740,'Musculoskeletal'),(800,1000,'Injury')]:
        if lo<=v<hi: return n
    return 'Other'
for col in ['diag_1','diag_2','diag_3']: d[col]=d[col].map(icd_group)
d['age_num']=d.age.str.extract(r'\[(\d+)-')[0].astype(float)+5
for col in ['admission_type_id','discharge_disposition_id','admission_source_id']: d[col]=d[col].astype(str)

drop=['encounter_id','patient_nbr','weight','examide','citoglipton','readmitted','y30','yany','age']
X=d.drop(columns=drop)
cat=X.select_dtypes(exclude='number').columns.tolist(); num=X.select_dtypes('number').columns.tolist()
X[cat]=X[cat].fillna('Missing').astype(str)
print(len(cat),'categorical,',len(num),'numeric features;', len(X),'rows')
print('prevalence <30: %.3f   any readmit: %.3f'%(d.y30.mean(),d.yany.mean()))

## Patient-grouped train/test split
20% of *patients* are held out for the final test. Model comparison uses grouped CV on the training part only.

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr,te=next(gss.split(X,groups=groups))
assert not set(groups[tr])&set(groups[te])
Xtr,Xte=X.iloc[tr],X.iloc[te]; gtr=groups[tr]
print(len(tr),len(te))

In [ ]:
def prep(scale):
    ohe=OneHotEncoder(handle_unknown='ignore',min_frequency=20,sparse_output=scale)
    numt=Pipeline([('i',SimpleImputer(strategy='median'))]+([('s',StandardScaler())] if scale else []))
    return ColumnTransformer([('c',ohe,cat),('n',numt,num)])
models={
 'Baseline (prior)':Pipeline([('p',prep(True)),('m',DummyClassifier(strategy='prior'))]),
 'Logistic regression':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.05,max_iter=2000))]),
 'Gradient boosting':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=250,max_leaf_nodes=15,
        min_samples_leaf=100,l2_regularization=1.0,random_state=SEED))]),
}
def evaluate(target):
    y=d[target].values; ytr,yte=y[tr],y[te]; rows=[]; fitted={}
    cv=GroupKFold(n_splits=5)
    for n,m in models.items():
        oof=cross_val_predict(m,Xtr,ytr,groups=gtr,cv=cv,method='predict_proba')[:,1]
        m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; fitted[n]=p
        rows.append(dict(model=n,cv_auc=roc_auc_score(ytr,oof),cv_pr=average_precision_score(ytr,oof),
                         test_auc=roc_auc_score(yte,p),test_pr=average_precision_score(yte,p)))
    return pd.DataFrame(rows).set_index('model').round(3),fitted,yte

## A) Readmitted within 30 days (`<30`) vs. not

In [ ]:
resA,predA,yteA=evaluate('y30'); print('Test prevalence: %.3f'%yteA.mean()); resA

## B) Readmitted at all (`<30` or `>30`) vs. `NO`

In [ ]:
resB,predB,yteB=evaluate('yany'); print('Test prevalence: %.3f'%yteB.mean()); resB

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(11,9))
for r,(name,pr,yt) in enumerate([('A: <30d',predA,yteA),('B: any',predB,yteB)]):
    for n,p in pr.items():
        if 'Baseline' in n: continue
        f,t,_=roc_curve(yt,p); ax[r,0].plot(f,t,label=n)
        pp,rr,_=precision_recall_curve(yt,p); ax[r,1].plot(rr,pp,label=n)
    ax[r,0].plot([0,1],[0,1],'k--'); ax[r,0].set(title=f'ROC - {name}',xlabel='FPR',ylabel='TPR')
    ax[r,1].axhline(yt.mean(),color='k',ls='--'); ax[r,1].set(title=f'Precision-Recall - {name}',xlabel='Recall',ylabel='Precision')
    ax[r,0].legend()
plt.tight_layout(); plt.show()

## Is the ranking useful? Lift by risk decile (task A, gradient boosting, test set)

In [ ]:
t=pd.DataFrame({'y':yteA,'p':predA['Gradient boosting']})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
g=t.groupby('decile',observed=True).agg(readmit_rate=('y','mean'),n=('y','size'))
g['lift']=g.readmit_rate/yteA.mean(); print(g.round(3))
g.readmit_rate.plot.bar(title='30-day readmission rate by predicted-risk decile'); plt.axhline(yteA.mean(),color='k',ls='--'); plt.show()

## What drives the predictions? (permutation importance, task A, test set)

In [ ]:
m=models['Gradient boosting']
m.fit(Xtr,d.y30.values[tr])
pi=permutation_importance(m,Xte,yteA,scoring='roc_auc',n_repeats=5,random_state=SEED,n_jobs=1)
imp=pd.Series(pi.importances_mean,index=X.columns).sort_values(ascending=False)
imp.head(12).iloc[::-1].plot.barh(title='Drop in test AUC when feature is shuffled'); plt.show()

## Conclusions (numbers from the run I did; re-run to reproduce)
- **Yes, but only modestly.** On unseen patients, gradient boosting reaches test ROC-AUC ~0.66 / PR-AUC ~0.21 for `<30`-day readmission (prevalence 11%), and ROC-AUC ~0.69 for any readmission (prevalence 47%). Chance is 0.50.
- The model ranks risk usefully: the top decile has a 30-day readmission rate ~2.3x the average and the bottom decile ~0.3x. It cannot classify individual patients reliably, so use it to prioritise follow-up, not to make hard decisions.
- Most signal comes from prior inpatient visits (`number_inpatient`) and discharge disposition, followed by primary diagnosis group, number of diagnoses and age.
- Caveats: administrative data only; the split is by patient but not by time, so drift is untested; the decision threshold depends on the cost of a missed readmission vs. an unnecessary intervention; the notebook was saved without outputs (run all cells to populate them).